# Lesson 10a: Continuous Control Theory

**Learning Objectives:**
- Understand continuous action spaces and deterministic policies
- Master Deep Deterministic Policy Gradient (DDPG) algorithm
- Learn Twin Delayed DDPG (TD3) improvements
- Study Soft Actor-Critic (SAC) with entropy regularization
- Compare on-policy vs off-policy continuous control methods
- Understand actor-critic architectures for continuous domains

**Prerequisites:** Policy Gradients (8a), PPO (9a), Function Approximation (6a)

**References:**
- Silver et al. (2014) - Deterministic Policy Gradient
- Lillicrap et al. (2015) - DDPG
- Fujimoto et al. (2018) - TD3
- Haarnoja et al. (2018) - SAC
- Sutton & Barto Chapter 13

## 1. Introduction to Continuous Control

### 1.1 Continuous vs Discrete Action Spaces

**Discrete Actions:**
- Finite set: $\mathcal{A} = \{a_1, a_2, ..., a_n\}$
- Policy: $\pi(a|s) \in [0,1]$, $\sum_a \pi(a|s) = 1$
- Examples: Atari games, board games, navigation (4 directions)

**Continuous Actions:**
- Continuous space: $\mathcal{A} \subseteq \mathbb{R}^d$
- Often bounded: $a \in [a_{\min}, a_{\max}]^d$
- Examples: Robot joint angles, vehicle steering/throttle, quadrotor control

### 1.2 Why Continuous Control is Challenging

**Problems with discretization:**
1. **Curse of dimensionality**: 10 dimensions × 10 values = $10^{10}$ discrete actions
2. **Loss of precision**: Can't represent smooth control signals
3. **Exploration difficulty**: Large discrete action spaces hard to explore

**Solution approaches:**
1. **Stochastic policies**: Model $\pi_\theta(a|s)$ as Gaussian $\mathcal{N}(\mu_\theta(s), \sigma_\theta(s))$
   - Used in: REINFORCE, A2C, PPO
   - Pro: Natural exploration
   - Con: High variance, sample inefficient

2. **Deterministic policies**: $a = \mu_\theta(s)$ with separate exploration noise
   - Used in: DDPG, TD3
   - Pro: Sample efficient with replay buffer
   - Con: Need explicit exploration strategy

3. **Maximum entropy RL**: Optimize $\mathbb{E}[\sum_t r_t + \alpha H(\pi)]$
   - Used in: SAC
   - Pro: Automatic exploration, robust
   - Con: More complex, extra hyperparameter

## 2. Deterministic Policy Gradient (DPG)

### 2.1 Motivation

**Standard Policy Gradient:**
$$\nabla_\theta J(\pi_\theta) = \mathbb{E}_{s \sim \rho^\pi, a \sim \pi_\theta}[\nabla_\theta \log \pi_\theta(a|s) Q^\pi(s,a)]$$

Problem: Requires integrating over continuous action space:
$$\nabla_\theta J = \int_\mathcal{S} \rho^\pi(s) \int_\mathcal{A} \nabla_\theta \pi_\theta(a|s) Q^\pi(s,a) \, da \, ds$$

High variance when sampling from $\pi_\theta$!

### 2.2 Deterministic Policy Gradient Theorem

**Silver et al. (2014)** proved:

For deterministic policy $\mu_\theta: \mathcal{S} \to \mathcal{A}$:

$$\nabla_\theta J(\mu_\theta) = \mathbb{E}_{s \sim \rho^\mu}[\nabla_\theta \mu_\theta(s) \nabla_a Q^\mu(s,a)|_{a=\mu_\theta(s)}]$$

**Key insight:**
- No integration over actions!
- Chain rule: gradient through policy, then through Q-function
- Much lower variance than stochastic policy gradient

### 2.3 Derivation (Sketch)

Start with performance objective:
$$J(\mu_\theta) = \mathbb{E}_{s_0}[V^\mu(s_0)]$$

Expand using Bellman:
$$V^\mu(s) = Q^\mu(s, \mu_\theta(s))$$

Take gradient:
$$\nabla_\theta V^\mu(s) = \nabla_\theta Q^\mu(s, \mu_\theta(s))$$

Chain rule:
$$= \nabla_\theta \mu_\theta(s) \nabla_a Q^\mu(s,a)|_{a=\mu_\theta(s)} + \nabla_\theta Q^\mu(s, \mu_\theta(s))$$

The second term expands recursively and contributes to the state distribution $\rho^\mu$.

After careful expansion (see Silver et al. 2014):
$$\nabla_\theta J(\mu_\theta) = \int_\mathcal{S} \rho^\mu(s) \nabla_\theta \mu_\theta(s) \nabla_a Q^\mu(s,a)|_{a=\mu_\theta(s)} \, ds$$

$$= \mathbb{E}_{s \sim \rho^\mu}[\nabla_\theta \mu_\theta(s) \nabla_a Q^\mu(s,a)|_{a=\mu_\theta(s)}]$$

## 3. Deep Deterministic Policy Gradient (DDPG)

### 3.1 Algorithm Overview

**Lillicrap et al. (2015)** combined DPG with DQN techniques:

**Key components:**
1. **Actor network**: $\mu_\theta(s)$ (deterministic policy)
2. **Critic network**: $Q_w(s,a)$ (Q-function approximator)
3. **Target networks**: $\mu_{\theta'}$, $Q_{w'}$ for stable learning
4. **Replay buffer**: Store and sample transitions
5. **Exploration noise**: Add Ornstein-Uhlenbeck noise during training

### 3.2 DDPG Update Rules

**Critic update** (minimize TD error):
$$y_i = r_i + \gamma Q_{w'}(s_{i+1}, \mu_{\theta'}(s_{i+1}))$$
$$L(w) = \frac{1}{N} \sum_i (y_i - Q_w(s_i, a_i))^2$$
$$w \leftarrow w - \alpha_w \nabla_w L(w)$$

**Actor update** (maximize Q-value):
$$\nabla_\theta J \approx \frac{1}{N} \sum_i \nabla_\theta \mu_\theta(s_i) \nabla_a Q_w(s_i, a)|_{a=\mu_\theta(s_i)}$$
$$\theta \leftarrow \theta + \alpha_\theta \nabla_\theta J$$

**Target network updates** (soft updates):
$$\theta' \leftarrow \tau \theta + (1-\tau) \theta'$$
$$w' \leftarrow \tau w + (1-\tau) w'$$

where $\tau \ll 1$ (e.g., 0.001)

### 3.3 Exploration in DDPG

**Ornstein-Uhlenbeck (OU) process:**
$$d\mathcal{N}_t = \theta_{OU}(\mu_{OU} - \mathcal{N}_t)dt + \sigma_{OU}dW_t$$

Discretized:
$$\mathcal{N}_{t+1} = \mathcal{N}_t + \theta_{OU}(\mu_{OU} - \mathcal{N}_t) + \sigma_{OU}\epsilon_t$$

Action during exploration:
$$a_t = \mu_\theta(s_t) + \mathcal{N}_t$$

**Properties:**
- Temporally correlated noise (better for physical systems)
- Mean-reverting (returns to $\mu_{OU}$, often 0)
- Alternative: Simple Gaussian noise $\mathcal{N}(0, \sigma^2)$

## 4. Twin Delayed DDPG (TD3)

### 4.1 Problems with DDPG

**Fujimoto et al. (2018)** identified critical issues:

1. **Overestimation bias**: Q-learning maximizes over actions, leading to overestimation
   $$Q(s,a) \approx r + \gamma \max_{a'} Q(s', a')$$
   Approximation errors accumulate!

2. **Variance accumulation**: Target updates use $\mu_{\theta'}$ which depends on noisy Q-values

3. **Brittle to hyperparameters**: Small changes can cause divergence

### 4.2 TD3 Solutions

**Three key improvements:**

#### 4.2.1 Clipped Double Q-Learning

Maintain **two critic networks** $Q_{w_1}, Q_{w_2}$:

$$y = r + \gamma \min_{i=1,2} Q_{w_i'}(s', \mu_{\theta'}(s'))$$

Update both critics:
$$L(w_i) = \mathbb{E}[(y - Q_{w_i}(s,a))^2]$$

**Intuition**: Taking minimum prevents overestimation!

#### 4.2.2 Delayed Policy Updates

Update actor less frequently than critic:
- Update critic every step
- Update actor every $d$ steps (e.g., $d=2$)

**Intuition**: Let Q-values stabilize before updating policy!

#### 4.2.3 Target Policy Smoothing

Add noise to target policy to reduce variance:

$$y = r + \gamma \min_{i=1,2} Q_{w_i'}(s', \mu_{\theta'}(s') + \epsilon)$$

where $\epsilon \sim \text{clip}(\mathcal{N}(0, \sigma), -c, c)$

**Intuition**: Smooths Q-function by averaging over similar actions!

### 4.3 TD3 Algorithm

```
Initialize critics Q_w1, Q_w2, actor μ_θ, target networks
Initialize replay buffer D

for episode = 1 to M do:
    for t = 1 to T do:
        # Select action with exploration
        a = μ_θ(s) + ε, ε ~ N(0, σ)
        
        # Execute and store
        Execute a, observe r, s'
        D ← D ∪ {(s, a, r, s')}
        
        # Sample minibatch
        Sample N transitions from D
        
        # Compute target with smoothing
        ε ~ clip(N(0, σ_target), -c, c)
        a' = μ_θ'(s') + ε
        y = r + γ min(Q_w1'(s', a'), Q_w2'(s', a'))
        
        # Update critics
        w1 ← w1 - α∇_w1 (y - Q_w1(s,a))²
        w2 ← w2 - α∇_w2 (y - Q_w2(s,a))²
        
        # Delayed policy update
        if t mod d = 0:
            # Update actor
            θ ← θ + α∇_θ Q_w1(s, μ_θ(s))
            
            # Update targets
            θ' ← τθ + (1-τ)θ'
            w1' ← τw1 + (1-τ)w1'
            w2' ← τw2 + (1-τ)w2'
```

## 5. Soft Actor-Critic (SAC)

### 5.1 Maximum Entropy RL

**Haarnoja et al. (2018)** formulated entropy-regularized objective:

$$J(\pi) = \sum_{t=0}^T \mathbb{E}_{(s_t, a_t) \sim \rho_\pi}[r(s_t, a_t) + \alpha H(\pi(\cdot|s_t))]$$

where entropy $H(\pi(\cdot|s)) = -\mathbb{E}_{a \sim \pi}[\log \pi(a|s)]$

**Benefits:**
1. **Exploration**: High entropy = explore more actions
2. **Robustness**: Learn multiple solutions (safer)
3. **Stability**: Prevents premature convergence

### 5.2 Soft Value Functions

**Soft Q-function:**
$$Q^\pi(s_t, a_t) = r(s_t, a_t) + \mathbb{E}_{s_{t+1:T}, a_{t+1:T}}\left[\sum_{l=t+1}^T r(s_l, a_l) + \alpha H(\pi(\cdot|s_l))\right]$$

**Soft value function:**
$$V^\pi(s_t) = \mathbb{E}_{a_t \sim \pi}[Q^\pi(s_t, a_t) - \alpha \log \pi(a_t|s_t)]$$
$$= \mathbb{E}_{a_t \sim \pi}[Q^\pi(s_t, a_t)] + \alpha H(\pi(\cdot|s_t))$$

**Soft Bellman equation:**
$$Q^\pi(s_t, a_t) = r(s_t, a_t) + \gamma \mathbb{E}_{s_{t+1}}[V^\pi(s_{t+1})]$$

### 5.3 SAC Policy

**Stochastic policy** (unlike DDPG/TD3):

Output mean and log-std:
$$\mu_\theta(s), \log \sigma_\theta(s) = f_\theta(s)$$

Sample action (reparameterization trick):
$$\tilde{a}_t = \mu_\theta(s_t) + \sigma_\theta(s_t) \odot \epsilon_t, \quad \epsilon_t \sim \mathcal{N}(0, I)$$

Apply squashing (tanh):
$$a_t = \tanh(\tilde{a}_t)$$

Log-probability (with squashing correction):
$$\log \pi_\theta(a_t|s_t) = \log \mathcal{N}(\tilde{a}_t; \mu_\theta, \sigma_\theta) - \sum_{i=1}^d \log(1 - \tanh^2(\tilde{a}_{t,i}))$$

### 5.4 SAC Update Rules

**Critic update** (two Q-networks like TD3):
$$y = r + \gamma (\min_{i=1,2} Q_{w_i'}(s', a') - \alpha \log \pi_\theta(a'|s'))$$

where $a' \sim \pi_\theta(\cdot|s')$

$$L(w_i) = \mathbb{E}[(y - Q_{w_i}(s,a))^2]$$

**Actor update** (maximize entropy-augmented Q):
$$L(\theta) = \mathbb{E}_{s \sim D, a \sim \pi_\theta}[\alpha \log \pi_\theta(a|s) - Q_{w_1}(s,a)]$$

Gradient (reparameterization):
$$\nabla_\theta L = \nabla_\theta \alpha \log \pi_\theta(a|s) + (\nabla_a \alpha \log \pi_\theta(a|s) - \nabla_a Q(s,a))\nabla_\theta f_\theta(s)$$

**Temperature update** (automatic tuning of α):
$$L(\alpha) = \mathbb{E}_{a \sim \pi_\theta}[-\alpha \log \pi_\theta(a|s) - \alpha \bar{H}]$$

where $\bar{H}$ is target entropy (e.g., $-\dim(\mathcal{A})$)

## 6. Comparison of Methods

### 6.1 Algorithm Comparison Table

| Feature | DDPG | TD3 | SAC |
|---------|------|-----|-----|
| Policy type | Deterministic | Deterministic | Stochastic |
| Exploration | OU/Gaussian noise | Gaussian noise | Entropy maximization |
| Critic networks | 1 | 2 (twin) | 2 (twin) |
| Overestimation fix | None | Clipped double-Q | Clipped double-Q |
| Policy smoothing | No | Yes (target) | No (not needed) |
| Update frequency | Every step | Delayed actor | Every step |
| Temperature α | N/A | N/A | Auto-tuned |
| Sample efficiency | High | High | Highest |
| Stability | Moderate | High | Very high |
| Hyperparameter sensitivity | High | Moderate | Low |

### 6.2 When to Use Each

**DDPG:**
- Historical interest (foundational algorithm)
- Simple baseline
- Not recommended for production (TD3/SAC better)

**TD3:**
- When you want deterministic policy
- Robotics tasks (smooth control)
- Well-understood dynamics
- Prefer simplicity over entropy regularization

**SAC:**
- **Default choice for continuous control** (2025 standard)
- When exploration is critical
- Need robustness to hyperparameters
- Want automatic exploration scheduling
- Complex/stochastic environments

### 6.3 Performance Benchmarks

**MuJoCo benchmarks** (Fujimoto 2018, Haarnoja 2018):

```
HalfCheetah-v2 (1M steps):
  DDPG:  ~9,000
  TD3:   ~10,000
  SAC:   ~11,000

Ant-v2 (1M steps):
  DDPG:  ~1,000 (unstable)
  TD3:   ~4,500
  SAC:   ~5,500

Humanoid-v2 (1M steps):
  DDPG:  ~500 (very unstable)
  TD3:   ~5,000
  SAC:   ~6,000+
```

**Key takeaway**: SAC > TD3 > DDPG in both performance and stability

## 7. Theoretical Foundations

### 7.1 Policy Improvement in Continuous Spaces

**Deterministic policy gradient theorem** (Silver et al. 2014):

If $\mu_\theta$ is differentiable, then:
$$\nabla_\theta J(\mu_\theta) = \mathbb{E}[\nabla_\theta \mu_\theta(s) \nabla_a Q^\mu(s,a)|_{a=\mu_\theta(s)}]$$

is a policy improvement direction!

**Proof sketch:**
- Bellman equation: $V^\mu(s) = Q^\mu(s, \mu_\theta(s))$
- If we move $\mu_\theta$ in direction of $\nabla_a Q$, value increases
- Chain rule connects policy parameters to Q-gradient

### 7.2 Convergence Properties

**DDPG convergence** (not guaranteed!):
- Uses function approximation for both policy and Q
- No convergence guarantees (deadly triad: FA + off-policy + bootstrapping)
- Target networks help but don't solve fundamental issue

**TD3 improvements:**
- Clipped double-Q reduces overestimation bias
- Delayed updates reduce variance
- Empirically more stable, but no formal guarantees

**SAC convergence:**
- Entropy regularization improves exploration
- Soft policy iteration has formal convergence proof (tabular case)
- Function approximation case: empirically very stable

### 7.3 Bias-Variance Tradeoff

**Overestimation bias:**
$$\mathbb{E}[\max(Q_1, Q_2)] \geq \max(\mathbb{E}[Q_1], \mathbb{E}[Q_2])$$

**TD3 solution:**
$$\mathbb{E}[\min(Q_1, Q_2)] \leq \min(\mathbb{E}[Q_1], \mathbb{E}[Q_2])$$

Introduces slight underestimation bias, but reduces variance!

**Variance from target policy:**
- DDPG: $y = r + \gamma Q(s', \mu_{\theta'}(s'))$ - low variance
- SAC: $y = r + \gamma (Q(s', a') - \alpha \log \pi(a'|s'))$ - higher variance but better exploration

## 8. Advanced Topics

### 8.1 Action Space Parameterization

**Bounded actions:**
- Squashing: $a = a_{\max} \cdot \tanh(\mu_\theta(s))$
- Scaling: $a = a_{\min} + (a_{\max} - a_{\min}) \cdot \sigma(\mu_\theta(s))$

**Unbounded actions:**
- Direct output: $a = \mu_\theta(s)$
- Gaussian: $a \sim \mathcal{N}(\mu_\theta(s), \sigma_\theta(s))$

**Multi-dimensional actions:**
- Independent dimensions: $a_i \sim \mathcal{N}(\mu_i(s), \sigma_i(s))$
- Correlated: Full covariance matrix (expensive!)

### 8.2 Exploration Strategies

**Parameter space noise** (Plappert et al. 2017):
- Add noise to network weights instead of actions
- More consistent exploration
- Better for tasks requiring coordination

**Exploration bonuses:**
- Count-based: $r_{bonus} = \beta / \sqrt{N(s,a)}$
- Curiosity: $r_{bonus} = \eta \cdot \text{prediction\_error}$

**Entropy regularization** (SAC approach):
- Automatic exploration
- Decreases as learning progresses (if α is auto-tuned)

### 8.3 Network Architectures

**Actor network:**
```
Input: state (s)
Hidden: [400, 300] with ReLU
Output: action (a) with tanh squashing
```

**Critic network:**
```
Input: state (s) and action (a)
Option 1: Concatenate [s, a] → [400, 300] → Q(s,a)
Option 2: Process s → hidden, then [hidden, a] → [300] → Q(s,a)
```

**SAC uses Option 2** (state processed first, action added later)

### 8.4 Practical Considerations

**Hyperparameters:**
- Learning rates: $\alpha_{actor} = 10^{-3}$, $\alpha_{critic} = 10^{-3}$
- Discount: $\gamma = 0.99$
- Soft update: $\tau = 0.005$
- Replay buffer: $10^6$ transitions
- Batch size: 256
- TD3: Policy delay $d = 2$
- SAC: Initial $\alpha = 0.2$, auto-tune

**Normalization:**
- State normalization: $s' = (s - \mu_s) / \sigma_s$
- Reward scaling: $r' = r / r_{scale}$
- Critical for stability!

**Training tips:**
1. Start with random exploration (warmup period)
2. Use large replay buffer
3. Multiple updates per environment step
4. Monitor Q-value magnitude (shouldn't explode)
5. Track entropy (SAC) - should decrease over time

## 9. Mathematical Derivations

### 9.1 Soft Policy Iteration Convergence

**Soft Policy Evaluation:**
$$Q^{k+1}(s,a) = r(s,a) + \gamma \mathbb{E}[V^k(s')]$$
$$V^{k+1}(s) = \mathbb{E}_{a \sim \pi}[Q^{k+1}(s,a) - \alpha \log \pi(a|s)]$$

**Soft Policy Improvement:**
$$\pi^{k+1} = \arg\min_{\pi' \in \Pi} D_{KL}(\pi'(\cdot|s) \| \frac{\exp(Q^{k+1}(s,\cdot)/\alpha)}{Z^{k+1}(s)})$$

Solution (Boltzmann policy):
$$\pi^{k+1}(a|s) = \frac{\exp(Q^{k+1}(s,a)/\alpha)}{Z^{k+1}(s)}$$

**Convergence:** This iteration converges to optimal entropy-regularized policy!

### 9.2 Reparameterization Trick for SAC

Want to compute $\nabla_\theta \mathbb{E}_{a \sim \pi_\theta}[f(s,a)]$

**Problem:** Can't backprop through sampling!

**Solution:** Reparameterize:
$$a = \mu_\theta(s) + \sigma_\theta(s) \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0,I)$$

Now:
$$\nabla_\theta \mathbb{E}_{\epsilon}[f(s, \mu_\theta(s) + \sigma_\theta(s) \odot \epsilon)] = \mathbb{E}_{\epsilon}[\nabla_\theta f(s, \mu_\theta(s) + \sigma_\theta(s) \odot \epsilon)]$$

Can backprop through $\mu_\theta$ and $\sigma_\theta$!

### 9.3 Squashing Function Correction

After sampling $\tilde{a} \sim \mathcal{N}(\mu, \sigma)$, apply $a = \tanh(\tilde{a})$

**Log-probability correction:**
$$\log \pi(a|s) = \log \mathcal{N}(\tilde{a}; \mu, \sigma) - \log \frac{da}{d\tilde{a}}$$

Since $a = \tanh(\tilde{a})$:
$$\frac{da}{d\tilde{a}} = 1 - \tanh^2(\tilde{a}) = 1 - a^2$$

For multi-dimensional:
$$\log \pi(a|s) = \sum_{i=1}^d [\log \mathcal{N}(\tilde{a}_i; \mu_i, \sigma_i) - \log(1 - a_i^2)]$$

## 10. Implementation Considerations

### 10.1 Numerical Stability

**Log-std parametrization:**
- Don't output $\sigma$ directly (can be negative!)
- Output $\log \sigma$, then $\sigma = \exp(\log \sigma)$
- Clip: $\log \sigma \in [-20, 2]$ to prevent numerical issues

**Gradient clipping:**
```python
torch.nn.utils.clip_grad_norm_(parameters, max_norm=1.0)
```

**Entropy computation:**
- Use log-sum-exp trick to avoid overflow
- Add small epsilon ($10^{-6}$) to logs

### 10.2 Replay Buffer Design

**Standard buffer:**
```python
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, s, a, r, s_next, done):
        self.buffer.append((s, a, r, s_next, done))
    
    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)
```

**Prioritized replay** (optional enhancement):
- Sample transitions with probability $\propto |\delta|^\omega$
- TD error $\delta = r + \gamma Q(s',a') - Q(s,a)$
- More sample efficient but more complex

### 10.3 Debugging Tips

**Monitor these metrics:**
1. **Q-values**: Should be reasonable (not too large)
2. **Policy entropy** (SAC): Should decrease over time
3. **TD error**: Should decrease as learning progresses
4. **Actor/critic loss**: Should be relatively stable
5. **Gradient norms**: Shouldn't explode

**Common issues:**
- Q-values explode → reduce learning rate, check target updates
- Policy doesn't improve → check gradient flow, increase exploration
- Unstable training → normalize states/rewards, use gradient clipping
- Poor exploration → increase noise (DDPG/TD3) or α (SAC)

## Summary

### Key Takeaways

1. **Continuous control** requires different approaches than discrete action spaces

2. **Deterministic Policy Gradient** theorem enables efficient continuous control:
   $$\nabla_\theta J = \mathbb{E}[\nabla_\theta \mu_\theta(s) \nabla_a Q(s,a)|_{a=\mu_\theta(s)}]$$

3. **DDPG** combines DPG with DQN techniques (replay buffer, target networks)

4. **TD3** improves DDPG with:
   - Clipped double Q-learning (reduce overestimation)
   - Delayed policy updates (reduce variance)
   - Target policy smoothing (improve stability)

5. **SAC** uses maximum entropy RL:
   - Stochastic policy with automatic exploration
   - Most robust and sample efficient (2025 standard)
   - Auto-tuned temperature parameter

6. **Algorithm choice:**
   - Default: **SAC** (best overall)
   - Deterministic control: **TD3**
   - Historical baseline: DDPG

### Next Steps

- **Lesson 10b**: Implement DDPG, TD3, and SAC from scratch
- **Lesson 11**: Model-based RL (planning with learned models)
- **Lesson 12**: Exploration strategies (curiosity, intrinsic motivation)

### Further Reading

1. Silver et al. (2014) - "Deterministic Policy Gradient Algorithms"
2. Lillicrap et al. (2015) - "Continuous control with deep reinforcement learning" (DDPG)
3. Fujimoto et al. (2018) - "Addressing Function Approximation Error in Actor-Critic Methods" (TD3)
4. Haarnoja et al. (2018) - "Soft Actor-Critic: Off-Policy Maximum Entropy Deep RL"
5. Haarnoja et al. (2019) - "Soft Actor-Critic Algorithms and Applications"
6. Sutton & Barto (2018) - Chapter 13: Policy Gradient Methods

## Exercises

### Theoretical Questions

1. **Derive** the deterministic policy gradient theorem from first principles

2. **Prove** that clipped double Q-learning reduces overestimation bias

3. **Explain** why target policy smoothing in TD3 improves stability

4. **Derive** the soft Bellman equation for maximum entropy RL

5. **Show** that the Boltzmann policy is the optimal solution for soft policy improvement

### Computational Exercises

6. **Compare** exploration strategies: OU noise vs Gaussian noise vs entropy regularization

7. **Implement** a minimal DDPG agent (without target networks) and analyze instability

8. **Ablation study**: Remove each TD3 improvement and measure performance degradation

9. **Tune** SAC temperature parameter manually vs auto-tuning - compare results

10. **Benchmark** DDPG vs TD3 vs SAC on MuJoCo environments

### Practical Applications

11. **Design** a continuous control task for a 2D robot arm and solve with SAC

12. **Analyze** the effect of replay buffer size on sample efficiency

13. **Implement** prioritized replay for TD3 and measure improvement

14. **Compare** state normalization strategies and their impact on training

15. **Debug** a failing continuous control training run using monitoring techniques